In [225]:
import torch
import torch.nn.functional as F

In [226]:
words = open('names.txt','r').read().splitlines()
chs = sorted(list(set(''.join(words))))
stoi = {c:i+1 for i,c in enumerate(chs)}
stoi['.'] = 0
itos = {stoi[c]:c for c in stoi}


In [227]:
N = torch.zeros(27,27)
for word in words:
    pW = '.' + word + '.'
    for ch1, ch2 in zip(pW, pW[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1][ix2]+=1


In [228]:
N += 1
N = N.float()
P = N/N.sum(dim=1, keepdim=True)

In [229]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    ix = 0
    word = []
    while True:
        ix = torch.multinomial(P[ix], num_samples = 1, generator = g).item()
        if ix==0:
            break
        ch = itos[ix]
        word.append(ch)
    print("".join(word))
        



cexze
momasurailezitynn
konimittain
llayn
ka


In [230]:
logP = torch.log(P)
logSum = 0
totalPairs = 0
for word in words:
     pW = '.' + word + '.'
     for ch1, ch2 in zip(pW, pW[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        logSum+=logP[ix1, ix2]
        totalPairs+=1
loss = -logSum/totalPairs
loss.item()

2.4543561935424805

In [231]:
xs = []
ys = []
total = 0
for word in words:
    pW = '.' + word + '.'
    for ch1, ch2 in zip(pW, pW[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
        total+=1

In [232]:
v = F.one_hot(torch.tensor(xs), num_classes=27).float()
g = torch.Generator().manual_seed(2147483647)
W = torch.randn(27,27, generator=g, requires_grad=True)

In [233]:
for i in range(100):

    logits = v @ W
    counts = logits.exp()
    probs = counts/counts.sum(1, keepdim=True)
    logProb = torch.log(probs[torch.arange(total), ys])
    loss = -torch.mean(logProb) + 0.01 * (W**2).mean()

    W.grad = None

    
    loss.backward()

    W.data -= 50 * W.grad

    print(loss.item())
    



3.7686190605163574
3.3788068294525146
3.161090850830078
3.027186155319214
2.9344842433929443
2.8672313690185547
2.816654682159424
2.777146577835083
2.745253801345825
2.7188303470611572
2.696505546569824
2.6773719787597656
2.6608052253723145
2.6463515758514404
2.633665084838867
2.622471570968628
2.6125476360321045
2.6037068367004395
2.595794916152954
2.5886809825897217
2.582256317138672
2.5764293670654297
2.5711238384246826
2.566272735595703
2.5618226528167725
2.5577263832092285
2.5539445877075195
2.550442695617676
2.5471925735473633
2.5441696643829346
2.5413525104522705
2.538722038269043
2.536262035369873
2.5339581966400146
2.531797409057617
2.5297679901123047
2.527860164642334
2.5260636806488037
2.5243709087371826
2.522773265838623
2.521263837814331
2.519836664199829
2.5184857845306396
2.517204999923706
2.515990734100342
2.5148372650146484
2.5137410163879395
2.512697696685791
2.511704921722412
2.5107581615448
2.509854793548584
2.5089924335479736
2.5081686973571777
2.507380247116089
2.

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    ix = 0
    word = []
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        probs = counts/counts.sum(1, keepdim=True)
        ix = torch.multinomial(probs, num_samples = 1, generator = g).item()
        word.append(itos[ix])
        if ix==0:
            break
    print("".join(word))
        

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[-2.4338,  2.0560,  0.8380,  1.0041,  1.0957,  0.9969, -0.2957,  0.1715,
          0.4373,  0.0485,  1.4559,  1.6577,  1.0233,  1.5027,  0.7075, -0.3512,
         -0.0876, -1.7033,  1.0651,  1.2914,  0.8396, -1.7447, -0.3974, -0.5944,
         -1.3523, -0.0500,  0.4981]], grad_fn=<MmBackward0>)
tensor([[0.0877, 7.8145, 2.3118, 2.7294, 2.9914, 2.7099, 0.7440, 1.1870, 1.5485,
         1.0497, 4.2883, 5.2474, 2.7825, 4.4939, 2.0289, 0.7039, 0.9161, 0.1821,
         2.9011, 3.6379, 2.3154, 0.1747, 0.6721, 0.5519, 0.2586, 0.9513, 1.6456]],
       grad_fn=<ExpBackward0>)
tensor([[0.0015, 0.1373, 0.0406, 0.0479, 0.0525, 0.0476, 0.0131, 0.0209, 0.0272,
         0.0184, 0.0753, 0.0922, 0.0489, 0.0789, 0.0356, 0.0124, 0.0161, 0.0032,
         0.0510, 0.0639, 0.0407, 0.0031, 0.0118, 0.0097, 0.0045, 0.0167, 0.0289]],
       grad_fn=<DivBackward0>)
3
tensor([[0., 0